# 1.6 VAF distributions by sample frequency and AltDepth

This notebook plots the distribution of VAF (`Sample.AltFrac`) separately for **DeepVariant** and **Mutect2**.

For every combination of:

- sample-frequency cutoff: **5% or 10%**
- AltDepth split: **2, 3, 5, 7, or 10**

one Gaussian KDE figure is produced with six curves:

1. all valid AltDepth values and all sample frequencies;
2. AltDepth below the selected threshold, using all sample frequencies;
3. sample frequency at or below the cutoff and AltDepth below the threshold;
4. sample frequency at or below the cutoff and AltDepth at or above the threshold;
5. sample frequency above the cutoff and AltDepth below the threshold;
6. sample frequency above the cutoff and AltDepth at or above the threshold.

The x-axis is always VAF (`Sample.AltFrac`). No VAF cutoff is applied in this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

run_label = "Run2"

input_csv = Path(
    "/home/donetski/Notebooks/OutputFiles/04_qc_checking_on_target/"
    "04_Run2_per_variant_target_status_FULL.csv"
)

output_dir = (
    Path("/home/donetski/Notebooks/OutputFiles/")
    / "NEW_1.6_vaf_sample_frequency_altdepth_kde"
    / run_label.lower()
)

output_prefix = "1.6"

filter_to_pass = True
filter_to_on_target = True
exclude_altdepth_zero = True

callers = ["DeepVariant", "Mutect2"]

sample_frequency_cutoffs = [0.05, 0.10]
altdepth_thresholds = [2, 3, 5, 7, 10]

# Multiplies SciPy's automatically selected KDE bandwidth.
bw_adjust = 2

vaf_xmin = 0
vaf_xmax = 1.05
x_grid_points = 500

# Set to True to display every figure inside the notebook.
# False keeps the notebook smaller while still saving every PNG.
show_plots = False

# This can create a large CSV, so it is off by default.
save_frequency_annotated_csv = False

TARGET_GENES = [
    "ATM", "BARD1", "BRCA1", "BRCA2", "CDH1", "CDKN2A", "CHEK2", "MLH1",
    "MSH2", "MSH6", "PALB2", "PMS2", "PTEN", "RAD51C", "RAD51D", "TP53",
]

output_dir.mkdir(parents=True, exist_ok=True)
print("Output folder:", output_dir)

## Load and apply the same base QC filters as notebook 1.3

The numeric plotting columns are converted to numbers. Nonnumeric values become missing.

By default:

- only `FILTER == "PASS"` rows are retained;
- only on-target rows are retained;
- rows with `Sample.AltDepth == 0` are excluded from plotting.

With the default zero-depth exclusion, the `AltDepth < 2` group contains AltDepth equal to 1.

In [ ]:
def read_csv_auto(path):
    try:
        return pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin1")


def require_columns(input_df, columns):
    missing = [column for column in columns if column not in input_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")


df = read_csv_auto(input_csv)

required_columns = [
    "Sample.ID", "caller", "Gene",
    "Sample.AltFrac", "Sample.AltDepth", "Sample.Depth",
    "Chr", "Start", "REF", "ALT",
]
require_columns(df, required_columns)

numeric_columns = ["Sample.AltFrac", "Sample.AltDepth", "Sample.Depth", "Start"]
if "End" in df.columns:
    numeric_columns.append("End")

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

df_qc = df.copy()

if filter_to_pass:
    require_columns(df_qc, ["FILTER"])
    df_qc = df_qc[df_qc["FILTER"].eq("PASS")].copy()

if filter_to_on_target:
    require_columns(df_qc, ["on_target"])
    on_target_mask = (
        df_qc["on_target"]
        .astype(str)
        .str.lower()
        .isin(["true", "1", "yes"])
    )
    df_qc = df_qc[on_target_mask].copy()

# Keep the pre-AltDepth-filter sample totals for the frequency denominator.
caller_sample_totals = (
    df_qc.groupby("caller")["Sample.ID"]
    .nunique()
    .rename("total_samples_for_caller")
    .reset_index()
)

if exclude_altdepth_zero:
    df_plot = df_qc[df_qc["Sample.AltDepth"] >= 1].copy()
else:
    df_plot = df_qc.copy()

print("Input rows:", f"{len(df):,}")
print("Rows after PASS/on-target filters:", f"{len(df_qc):,}")
print("Rows available for plotting:", f"{len(df_plot):,}")
print("Rows excluded because AltDepth = 0:", f"{len(df_qc) - len(df_plot):,}")
print()
print(caller_sample_totals)

## Calculate sample frequency separately for each caller

For each exact variant and caller:

1. duplicate rows from the same sample are collapsed;
2. `sample_count` is the number of unique samples containing that variant;
3. `sample_fraction` is `sample_count / total samples for that caller`.

The calculation is performed before making the all-variant and 16-gene analysis sets, so a variant receives the same sample-frequency value in both sets.

In [ ]:
variant_cols = ["Chr", "Start"]
if "End" in df_plot.columns:
    variant_cols.append("End")
variant_cols += ["REF", "ALT"]

frequency_key = ["caller"] + variant_cols

one_row_per_sample_variant = df_plot.drop_duplicates(
    subset=["caller", "Sample.ID"] + variant_cols
)

frequency_df = (
    one_row_per_sample_variant
    .groupby(frequency_key, dropna=False)
    .agg(sample_count=("Sample.ID", "nunique"))
    .reset_index()
)

frequency_df = frequency_df.merge(
    caller_sample_totals,
    on="caller",
    how="left",
    validate="many_to_one",
)

frequency_df["sample_fraction"] = (
    frequency_df["sample_count"]
    / frequency_df["total_samples_for_caller"]
)
frequency_df["sample_percent"] = (
    frequency_df["sample_fraction"] * 100
).round(2)

df_plot = df_plot.merge(
    frequency_df,
    on=frequency_key,
    how="left",
    validate="many_to_one",
)

if save_frequency_annotated_csv:
    frequency_csv = (
        output_dir
        / f"{output_prefix}_{run_label}_variants_with_caller_sample_frequency.csv"
    )
    df_plot.to_csv(frequency_csv, index=False)
    print("Saved:", frequency_csv)

df_plot[
    ["Sample.ID", "caller", "Gene"]
    + variant_cols
    + [
        "sample_count",
        "total_samples_for_caller",
        "sample_fraction",
        "sample_percent",
    ]
].head()

## Create the two analysis sets

As in notebook 1.3, plots are generated for:

- `all_variants`: all rows remaining after the base QC filters;
- `target_16_genes`: only the 16 breast-cancer susceptibility genes.

Figures are organized as:

```text
run2/
  all_variants/
    figures/
      deepvariant/
        samplefreq_5pct/
        samplefreq_10pct/
      mutect2/
        samplefreq_5pct/
        samplefreq_10pct/
  target_16_genes/
    figures/
      ...
```

In [ ]:
target_gene_mask = (
    df_plot["Gene"]
    .astype(str)
    .str.upper()
    .isin(TARGET_GENES)
)

analysis_sets = {
    "all_variants": df_plot.copy(),
    "target_16_genes": df_plot[target_gene_mask].copy(),
}

for set_name, set_df in analysis_sets.items():
    print(f"{set_name}: {len(set_df):,} rows")

## Gaussian KDE plotting helpers

Every curve uses explicit `scipy.stats.gaussian_kde`.

`bw_adjust = 2` doubles SciPy's automatically selected bandwidth, matching the smoothing idea of:

```python
sns.kdeplot(..., bw_adjust=2)
```

All valid values in a group are used to fit its KDE. The curves are then evaluated on the common VAF range from 0 to 1.05.

The two reference curves are dashed. The four joint sample-frequency/AltDepth groups are solid.

In [ ]:
def safe_name(value):
    return (
        str(value)
        .replace(" ", "_")
        .replace("/", "_")
        .replace(".", "p")
        .replace("<=", "le")
        .replace(">=", "ge")
        .replace(">", "gt")
        .lower()
    )


def percent_folder(value):
    return f"samplefreq_{int(round(value * 100))}pct"

def save_plot(path):
    plt.tight_layout(pad=0.6)
    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.05,
    )
    if show_plots:
        plt.show()

    plt.close()
    print("Saved:", path)


def plot_gaussian_kde(
    series,
    label,
    x_grid,
    linestyle="-",
    linewidth=1.8,
):
    values = pd.to_numeric(series, errors="coerce").dropna()

    # Gaussian KDE requires at least two non-identical values.
    if len(values) < 2 or values.nunique() < 2:
        return False

    kde = gaussian_kde(values)

    # SciPy equivalent of multiplying the automatic bandwidth by bw_adjust.
    kde.set_bandwidth(bw_method=kde.factor * bw_adjust)

    plt.plot(
        x_grid,
        kde(x_grid),
        label=label,
        linestyle=linestyle,
        linewidth=linewidth,
    )
    return True


def build_vaf_groups(caller_df, sample_freq_cutoff, altdepth_threshold):
    low_depth = caller_df["Sample.AltDepth"] < altdepth_threshold
    high_depth = caller_df["Sample.AltDepth"] >= altdepth_threshold
    low_frequency = caller_df["sample_fraction"] <= sample_freq_cutoff
    high_frequency = caller_df["sample_fraction"] > sample_freq_cutoff

    return [
        (
            "All variants",
            caller_df,
            "--",
            "all",
        ),
        (
            f"AltDepth < {altdepth_threshold}",
            caller_df[low_depth],
            "--",
            "low_depth_reference",
        ),
        (
            f"Freq ≤ {sample_freq_cutoff:.0%}, depth < {altdepth_threshold}",
            caller_df[low_frequency & low_depth],
            "-",
            "low_freq_low_depth",
        ),
        (
            f"Freq ≤ {sample_freq_cutoff:.0%}, depth ≥ {altdepth_threshold}",
            caller_df[low_frequency & high_depth],
            "-",
            "low_freq_high_depth",
        ),
        (
            f"Freq > {sample_freq_cutoff:.0%}, depth < {altdepth_threshold}",
            caller_df[high_frequency & low_depth],
            "-",
            "high_freq_low_depth",
        ),
        (
            f"Freq > {sample_freq_cutoff:.0%}, depth ≥ {altdepth_threshold}",
            caller_df[high_frequency & high_depth],
            "-",
            "high_freq_high_depth",
        ),
    ]
    

def plot_vaf_by_samplefreq_and_altdepth(
    caller_df,
    caller_name,
    set_name,
    sample_freq_cutoff,
    altdepth_threshold,
    figure_dir,
):
    groups = build_vaf_groups(
        caller_df,
        sample_freq_cutoff,
        altdepth_threshold,
    )

    x_grid = np.linspace(vaf_xmin, vaf_xmax, x_grid_points)

    #plt.figure(figsize=(10, 6))
    plt.figure(figsize=(5.5, 4.8))
    plotted_any = False
    group_summary = []

    for label, subset, linestyle, group_key in groups:
        plotted = plot_gaussian_kde(
            subset["Sample.AltFrac"],
            label,
            x_grid,
            linestyle=linestyle,
        )
        plotted_any |= plotted

        group_summary.append({
            "analysis_set": set_name,
            "caller": caller_name,
            "sample_frequency_cutoff": sample_freq_cutoff,
            "altdepth_threshold": altdepth_threshold,
            "group_key": group_key,
            "group_label": label,
            "n_rows": len(subset),
            "n_unique_samples": subset["Sample.ID"].nunique(),
            "n_unique_variants": subset.drop_duplicates(
                subset=variant_cols
            ).shape[0],
            "median_vaf": subset["Sample.AltFrac"].median(),
            "median_altdepth": subset["Sample.AltDepth"].median(),
            "kde_plotted": plotted,
        })

    if not plotted_any:
        plt.close()
        print(
            f"Skipped: {set_name}, {caller_name}, "
            f"sample frequency {sample_freq_cutoff:.0%}, "
            f"AltDepth {altdepth_threshold}"
        )
        return group_summary

    # Reference positions only; excluded from the legend to keep it readable.
    plt.axvline(0.5, linestyle=":", linewidth=1)
    plt.axvline(1.0, linestyle=":", linewidth=1)

    plt.xlim(vaf_xmin, vaf_xmax)
    plt.xlabel(
        "Variant allele frequency (VAF)",
        fontsize=13,
        labelpad=7,
    )
    
    plt.ylabel(
        "Density",
        fontsize=13,
        labelpad=7,
    )
    
    plt.xticks(np.arange(0.0, 1.01, 0.1),fontsize=11,)
    plt.yticks(fontsize=11)
    
    plt.title(
        f"{caller_name} VAF distributions\n"
        f"Sample frequency: {sample_freq_cutoff:.0%}; "
        f"AltDepth threshold: {altdepth_threshold}",
        fontsize=14,
        fontweight="bold",
        pad=9,
    )
    
    plt.legend(
        fontsize=8.5,
        loc="upper right",
        frameon=True,
        handlelength=2,
        borderpad=0.4,
        labelspacing=0.3,
    )

    # plt.xlim(vaf_xmin, vaf_xmax)
    # plt.xlabel("Sample.AltFrac (VAF)")
    # plt.ylabel("Density")
    # plt.title(
    #     f"{run_label} {set_name}: {caller_name} VAF Gaussian KDE\n"
    #     f"Sample-frequency cutoff {sample_freq_cutoff:.0%}; "
    #     f"AltDepth split < {altdepth_threshold} vs >= {altdepth_threshold}"
    # )
    # plt.legend(fontsize=8)

    caller_folder = safe_name(caller_name)
    cutoff_folder = percent_folder(sample_freq_cutoff)

    out_png = (
        figure_dir
        / caller_folder
        / cutoff_folder
        / (
            f"{output_prefix}_{run_label}_{set_name}_{caller_folder}_"
            f"vaf_kde_samplefreq_{int(round(sample_freq_cutoff * 100))}pct_"
            f"altdepth_split_{altdepth_threshold}.png"
        )
    )
    out_png.parent.mkdir(parents=True, exist_ok=True)

    save_plot(out_png)
    return group_summary

## Generate every requested figure

For each analysis set and caller, this loops through:

- sample-frequency cutoffs: 5% and 10%;
- AltDepth thresholds: 2, 3, 5, 7, and 10.

Expected total with two analysis sets:

```text
2 analysis sets × 2 callers × 2 sample-frequency cutoffs × 5 AltDepth thresholds
= 40 figures
```

A summary CSV records the number of rows, unique samples, unique variants, median VAF, and median AltDepth for every plotted group.

In [ ]:
summary_rows = []

expected_plot_count = (
    len(analysis_sets)
    * len(callers)
    * len(sample_frequency_cutoffs)
    * len(altdepth_thresholds)
)

print("Expected figures:", expected_plot_count)

for set_name, set_df in analysis_sets.items():
    figure_dir = output_dir / set_name / "figures"

    for caller_name in callers:
        caller_df = set_df[
            set_df["caller"].eq(caller_name)
        ].copy()

        if caller_df.empty:
            print(f"Skipped {set_name} / {caller_name}: no rows")
            continue

        for sample_freq_cutoff in sample_frequency_cutoffs:
            for altdepth_threshold in altdepth_thresholds:
                summary_rows.extend(
                    plot_vaf_by_samplefreq_and_altdepth(
                        caller_df=caller_df,
                        caller_name=caller_name,
                        set_name=set_name,
                        sample_freq_cutoff=sample_freq_cutoff,
                        altdepth_threshold=altdepth_threshold,
                        figure_dir=figure_dir,
                    )
                )

summary_df = pd.DataFrame(summary_rows)

summary_path = (
    output_dir
    / f"{output_prefix}_{run_label}_vaf_samplefreq_altdepth_group_summary.csv"
)
summary_df.to_csv(summary_path, index=False)

print()
print("Saved summary:", summary_path)
print("Summary rows:", f"{len(summary_df):,}")
summary_df.head(12)

## Optional checks

The first table counts how many figures were successfully represented in the summary.

The second table shows the row counts for each caller, sample-frequency cutoff, AltDepth threshold, and group. This is useful for spotting empty or very small groups before interpreting a KDE curve.

In [ ]:
plot_check = (
    summary_df[
        [
            "analysis_set",
            "caller",
            "sample_frequency_cutoff",
            "altdepth_threshold",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print("Requested figure combinations represented:", plot_check)
print("Expected figure combinations:", expected_plot_count)

summary_df[
    [
        "analysis_set",
        "caller",
        "sample_frequency_cutoff",
        "altdepth_threshold",
        "group_key",
        "n_rows",
        "n_unique_samples",
        "n_unique_variants",
        "kde_plotted",
    ]
].head(30)